In [ ]:
# ==============================================================================
# Google Colab用 ワードクラウド作成ツール
#
# 【使い方】
# 1. このコードをすべてコピーし、Google Colabの新しいセルに貼り付けます。
# 2. セルの左側にある「実行ボタン（▶）」を押します。
# 3. 必要なライブラリとフォントがインストールされた後、「ファイル選択」ボタンが表示されます。
# 4. 分析したい日本語のテキストファイル（.txt形式など）をアップロードしてください。
# ==============================================================================

# 1. 必要なライブラリと日本語フォントのインストール（初回実行時は少し時間がかかります）
!pip install -q wordcloud janome
!apt-get -y -q install fonts-ipafont-gothic

# 2. ツールのメインコード
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from janome.tokenizer import Tokenizer
from google.colab import files
import warnings
warnings.filterwarnings('ignore')

# Colabにインストールした日本語フォントのパス
font_path = '/usr/share/fonts/opentype/ipafont-gothic/ipagp.ttf'

def generate_wordcloud(text):
    print("\nテキストを解析しています...")

    # 除外したい単語（ストップワード）のリスト
    stop_words = [
        'する', 'ある', 'ない', 'いる', 'なる', 'れる', 'られる', 'おる', 'せる',
        'の', 'こと', 'もの', 'これ', 'それ', 'あれ', '私', '僕', '自分', 'ため', 'よう'
    ]

    tokenizer = Tokenizer()
    words = []

    # 形態素解析を行い、意味のある単語（名詞・形容詞）を抽出
    for token in tokenizer.tokenize(text):
        pos_parts = token.part_of_speech.split(',')
        pos1 = pos_parts[0] # 品詞大分類
        pos2 = pos_parts[1] # 品詞細分類
        base_form = token.base_form # 原形

        # 名詞の処理（数詞、代名詞、非自立語は除外）
        if pos1 == '名詞' and pos2 not in ['非自立', '代名詞', '数']:
            if base_form not in stop_words:
                words.append(base_form)

        # 形容詞の処理（非自立語は除外）
        elif pos1 == '形容詞' and pos2 != '非自立':
            if base_form not in stop_words:
                words.append(base_form)

    # 抽出した単語をスペース区切りで結合
    word_space = ' '.join(words)

    if not word_space:
        print("抽出できる単語がありませんでした。テキストを確認してください。")
        return

    print("ワードクラウドを生成しています...\n")

    # ワードクラウドの設定
    wordcloud = WordCloud(
        font_path=font_path,
        width=1200,               # 画像の幅
        height=800,               # 画像の高さ
        background_color='white', # 背景色（'black' にすると黒背景になります）
        colormap='viridis',       # 文字のカラーマップ（'plasma', 'inferno', 'magma'なども可能）
        max_words=200,            # 表示する最大単語数
        collocations=False        # 複合語の重複表示を防ぐ
    ).generate(word_space)

    # 画像の描画と表示
    plt.figure(figsize=(15, 10))
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off") # 軸を非表示にする
    plt.tight_layout(pad=0)
    plt.show()

# 3. 実行とファイルアップロードUI
print("="*60)
print("テキストデータをアップロードしてください（.txtファイル推奨）")
print("※アップロードダイアログが表示されない場合は、ブラウザの設定をご確認ください。")
print("="*60)

# ファイルアップロードの実行
uploaded = files.upload()

if uploaded:
    # アップロードされたファイルを処理
    for filename in uploaded.keys():
        print(f"\n『{filename}』のワードクラウドを作成します。")
        try:
            # テキストの読み込み（UTF-8でデコード）
            text_data = uploaded[filename].decode('utf-8')
            generate_wordcloud(text_data)
        except UnicodeDecodeError:
            print("文字コードのエラーが発生しました。Shift-JISで読み込みを試みます。")
            try:
                text_data = uploaded[filename].decode('shift_jis')
                generate_wordcloud(text_data)
            except Exception as e:
                 print(f"ファイルの読み込みに失敗しました: {e}")
                 print("UTF-8形式で保存されたテキストファイルをアップロードしてください。")
else:
    print("\nファイルがアップロードされませんでした。")
    print("サンプルの文章でワードクラウドを作成します。")

    sample_text = """
    人工知能（じんこうちのう、英: artificial intelligence、AI）とは、計算機（コンピュータ）によって実現される知能のこと。
    現在では機械学習やディープラーニングといった技術の急速な発展により、画像認識、自然言語処理、音声認識など様々な分野で幅広く活用されている。
    AIは私たちの生活を便利にする一方で、倫理的な課題や雇用の変化など、社会に大きな影響を与える技術としても注目を集めている。
    """
    generate_wordcloud(sample_text)